In [5]:
!pip install pandas numpy

In [1]:
import pandas as pd
import numpy as np

In [ ]:
df1 = pd.read_csv('generated_cot\\problem_ids_matched.csv')
df2 = pd.read_csv('generated_cot\\train_split_with_cot.csv')
df3 = pd.read_csv('generated_cot\\train_split_with_cot_18_04_2026.csv')

src_df = pd.read_csv("src\\train.csv")

In [9]:
len(df1), len(df2), len(df3), len(src_df)

(7830, 6558, 6558, 9500)

In [11]:
df1.head()

,id,prompt,answer,type,generated_cot
0,77602e0f,"In Alice's Wonderland, a secret bit manipulati...",11001101,bit_manipulation,We need to deduce the transformation by matchi...
1,8a057351,"In Alice's Wonderland, a secret bit manipulati...",01110111,bit_manipulation,We need to deduce the transformation by matchi...
2,14dc1dbb,"In Alice's Wonderland, a secret unit conversio...",10.25,unit_conversion,We need to find a conversion rule that maps th...
3,7af9007a,"In Alice's Wonderland, numbers are secretly co...",LX,numeral,We need to determine the conversion rule from ...
4,524cb5c6,"In Alice's Wonderland, a secret set of transfo...",$>>\,cryptarithm_deduce,We need to infer the transformation rule from ...


In [12]:
df2.head()

,id,prompt,answer,type,generated_cot
0,7a4063e6,"In Alice's Wonderland, a secret bit manipulati...",10010110,Bit Manipulation,"Each bit is marked CERTAIN, so we can directly..."
1,7e4ca5bc,"In Alice's Wonderland, numbers are secretly co...",LVI,Numeral Conversion,1. Analyze each example carefully to identify ...
2,71cd0e14,"In Alice's Wonderland, numbers are secretly co...",XXV,Numeral Conversion,1. **Analyze each example carefully to identif...
3,ce862776,"In Alice's Wonderland, a secret bit manipulati...",01000000,Bit Manipulation,"Each bit is marked CERTAIN, so we just take th..."
4,f05e77f3,"In Alice's Wonderland, secret encryption rules...",wizard follows inside wonderland,Text Encryption,Let’s decode it step by step.\n\n1. Build the ...


In [13]:
df3.head()

,id,prompt,answer,type,generated_cot
0,7a4063e6,"In Alice's Wonderland, a secret bit manipulati...",10010110,Bit Manipulation,"Each bit is marked CERTAIN, so we can directly..."
1,7e4ca5bc,"In Alice's Wonderland, numbers are secretly co...",LVI,Numeral Conversion,1. Analyze each example carefully to identify ...
2,71cd0e14,"In Alice's Wonderland, numbers are secretly co...",XXV,Numeral Conversion,1. **Analyze each example carefully to identif...
3,ce862776,"In Alice's Wonderland, a secret bit manipulati...",01000000,Bit Manipulation,"Each bit is marked CERTAIN, so we just take th..."
4,f05e77f3,"In Alice's Wonderland, secret encryption rules...",wizard follows inside wonderland,Text Encryption,Let’s decode it step by step.\n\n1. Build the ...


In [15]:
src_df.head()

,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret


In [17]:
import pandas as pd
import re
import json
from collections import Counter


MAX_COTS_PER_Q = 2
MIN_COT_LEN = 30

USE_TOT = True           # enable ToT-style samples
TOT_RATIO = 0.5          # % of samples using ToT

OUTPUT_FILE = "final_sft.jsonl"

import pandas as pd
import re
import json
import random

# ---------- CONFIG ----------
COT_FILES = ["generated_cot\\problem_ids_matched.csv", ]
GT_FILE =  "src\\train.csv"  

OUTPUT_FILE = "final_sft.jsonl"

MIN_COT_LEN = 30
MAX_COTS_PER_Q = 2

USE_TOT = True
TOT_RATIO = 0.4


# ---------- HELPERS ----------

def normalize_answer(ans):
    if ans is None:
        return None
    ans = str(ans).strip().lower()
    ans = ans.replace(",", "")
    return ans


def extract_answer_from_cot(cot):
    """Try boxed first, else fallback to last line"""
    match = re.search(r"\\boxed\{([^}]*)\}", str(cot))
    if match:
        return match.group(1).strip()
    
    # fallback (important for your data)
    lines = str(cot).strip().split("\n")
    return lines[-1].strip()


def clean_cot(cot):
    return str(cot).strip()


def remove_think_tags(text):
    return re.sub(r"</?think>", "", text)


def is_valid_cot(cot):
    if not cot:
        return False
    if len(cot) < MIN_COT_LEN:
        return False
    return True


def score_cot(cot):
    score = 0
    
    score += min(len(cot) / 100, 5)
    
    if "step" in cot.lower(): score += 1
    if "therefore" in cot.lower(): score += 1
    
    if "guess" in cot.lower(): score -= 2
    if "maybe" in cot.lower(): score -= 1
    
    return score


# ---------- ToT BUILDER ----------

def build_error_cot(best_cot, weak_cot, answer):
    best_clean = remove_think_tags(best_cot)
    weak_clean = remove_think_tags(weak_cot)

    return f"""<think>
Incorrect attempt:
{weak_clean}

This approach is incorrect because it leads to a wrong result.

Correct reasoning:
{best_clean}
</think>
\\boxed{{{answer}}}
"""


# ---------- LOAD DATA ----------

dfs = []
for f in COT_FILES:
    df = pd.read_csv(f)
    
    # rename for consistency
    df = df.rename(columns={
        "prompt": "question",
        "generated_cot": "cot"
    })
    
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)


# ---------- LOAD GT ----------

gt_df = pd.read_csv(GT_FILE)
gt_df = gt_df.rename(columns={"prompt": "question"})

gt_map = {
    q: normalize_answer(a)
    for q, a in zip(gt_df["question"], gt_df["answer"])
}


# ---------- GROUP ----------

grouped = df_all.groupby("question")["cot"].apply(list).reset_index()


# ---------- PROCESS ----------

final_data = []

stats = {
    "total": 0,
    "no_gt": 0,
    "invalid": 0,
    "wrong": 0,
    "final": 0
}

for _, row in grouped.iterrows():
    stats["total"] += 1
    
    q = row["question"]

    if q not in gt_map:
        stats["no_gt"] += 1
        continue

    true_ans = gt_map[q]

    cots = [clean_cot(c) for c in row["cot"] if is_valid_cot(c)]

    if len(cots) == 0:
        stats["invalid"] += 1
        continue

    valid_cots = []

    for cot in cots:
        pred = extract_answer_from_cot(cot)
        
        if pred is None:
            continue
        
        if normalize_answer(pred) == true_ans:
            valid_cots.append(cot)

    if len(valid_cots) == 0:
        stats["wrong"] += 1
        continue

    # sort by quality
    valid_cots = sorted(valid_cots, key=score_cot, reverse=True)
    valid_cots = valid_cots[:MAX_COTS_PER_Q]

    # ---------- BUILD OUTPUT ----------

    if USE_TOT and len(valid_cots) >= 2 and random.random() < TOT_RATIO:
        best = valid_cots[0]
        weak = valid_cots[-1]
        
        final_cot = build_error_cot(best, weak, true_ans)
        
        final_data.append({
            "messages": [
                {"role": "user", "content": q},
                {"role": "assistant", "content": final_cot}
            ]
        })
    else:
        for cot in valid_cots:
            final_data.append({
                "messages": [
                    {"role": "user", "content": q},
                    {"role": "assistant", "content": cot}
                ]
            })

    stats["final"] += 1


# ---------- SAVE ----------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

csv_rows = []

for item in final_data:
    question = item["messages"][0]["content"]
    answer = item["messages"][1]["content"]
    
    csv_rows.append({
        "question": question,
        "response": answer
    })

df_csv = pd.DataFrame(csv_rows)

CSV_OUTPUT = OUTPUT_FILE.replace(".jsonl", ".csv")
df_csv.to_csv(CSV_OUTPUT, index=False, encoding="utf-8")

print(f"Saved CSV → {CSV_OUTPUT}")


print("\n===== DONE =====")
print(f"Saved {len(final_data)} samples → {OUTPUT_FILE}")

print("\n===== STATS =====")
for k, v in stats.items():
    print(f"{k}: {v}")

Saved CSV → final_sft.csv

===== DONE =====
Saved 0 samples → final_sft.jsonl

===== STATS =====
total: 6171
no_gt: 0
invalid: 0
wrong: 6171
final: 0
